# Parallelization workflow with Pydantic AI

Parallelization runs independent subtasks concurrently, then aggregates results. This trades coordination complexity for significant speed improvements.

```mermaid
flowchart LR
    In([In]) --> LLM1["LLM Call 1"]
    In --> LLM2["LLM Call 2"]
    In --> LLM3["LLM Call 3"]
    LLM1 --> Aggregator["Aggregator"] 
    LLM2 --> Aggregator
    LLM3 --> Aggregator
    Aggregator --> Out([Out])
```

**Examples:**
- Evaluate multiple independent aspects of a text (safety, quality, relevance)
- Process user query and apply guardrails in parallel
- Generate multiple response candidates for comparison

In [ ]:
import nest_asyncio

nest_asyncio.apply()

## Setup

In [ ]:
import asyncio
from pprint import pprint

import logfire
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from pydantic_ai import Agent

load_dotenv()

logfire.configure()
logfire.instrument_pydantic_ai()

## Vanilla workflow

We run the same evaluator 3 times in parallel using `asyncio.gather`, then aggregate results.

Since Pydantic AI is async-native, parallel execution is straightforward.

In [ ]:
class Evaluation(BaseModel):
    explanation: str
    is_appropriate: bool


class AggregatedResults(BaseModel):
    summary: str
    is_appropriate: bool


evaluator = Agent(
    "openai:gpt-5-nano",
    output_type=Evaluation,
    system_prompt=(
        "You are an expert evaluator. Provided with a text, "
        "you will evaluate if it's appropriate for a general audience."
    ),
)

aggregator = Agent(
    "openai:gpt-5-nano",
    output_type=AggregatedResults,
    system_prompt=(
        "You are an expert evaluator. Provided with a list of evaluations, "
        "you will summarize them and provide a final evaluation."
    ),
)


async def run_workflow(text: str) -> AggregatedResults:
    # Run 3 evaluations in parallel
    tasks = [evaluator.run(f"Evaluate the following text: {text}") for _ in range(3)]
    evaluations = await asyncio.gather(*tasks)

    # Aggregate results
    aggregated = await aggregator.run(
        f"Summarize the following evaluations:\n\n"
        f"{[(e.output.explanation, e.output.is_appropriate) for e in evaluations]}"
    )
    return aggregated.output


output = await run_workflow(
    "Athletes should consume enhancing drugs to improve their performance."
)
pprint(output.model_dump())

## Exercise

Implement a workflow that generates three jokes in parallel and picks the funniest one.

In [ ]:
class FunniestJoke(BaseModel):
    id: int = Field(
        description="1-based index of the funniest joke",
        ge=1,
        le=3,
    )
    reasoning: str


joke_writer = Agent(
    "openai:gpt-5-nano",
    system_prompt=(
        "You are an expert joke writer. Write one short original joke "
        "about the requested topic."
    ),
)

joke_selector = Agent(
    "openai:gpt-5-nano",
    output_type=FunniestJoke,
    system_prompt="You are a comedy judge. Pick the funniest joke and explain why.",
)


async def run_joke_workflow(topic: str) -> dict:
    tasks = [joke_writer.run(f"Write a joke about {topic}.") for _ in range(3)]
    joke_results = await asyncio.gather(*tasks)
    jokes = [result.output for result in joke_results]

    selection = await joke_selector.run(
        "Pick the funniest joke from this list:\n\n"
        + "\n\n".join(f"{i + 1}. {joke}" for i, joke in enumerate(jokes))
    )

    return {
        "jokes": jokes,
        "funniest_joke": jokes[selection.output.id - 1],
        "reasoning": selection.output.reasoning,
    }


joke_result = await run_joke_workflow("software engineers")
for i, joke in enumerate(joke_result["jokes"], start=1):
    print(f"{i}. {joke}\n")
print("Winner:", joke_result["funniest_joke"])
print("Why:", joke_result["reasoning"])